# Notebook 01 : Data Cleaning & Analyse Exploratoire des Données (EDA)
**Projet** : Détection de Fraude aux Paiements par Carte Bancaire  


---

## 1. Importation des Bibliothèques & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration des graphiques
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
USD_TO_MAD = 10.0 # Taux de conversion fixe (1 USD = 10 DH)

## 2. Chargement des Données Brutes

In [ ]:
df_train = pd.read_csv("../data/fraudTrain.csv") if os.path.exists("../data/fraudTrain.csv") else pd.read_csv("data/fraudTrain.csv")
df_test = pd.read_csv("../data/fraudTest.csv") if os.path.exists("../data/fraudTest.csv") else pd.read_csv("data/fraudTest.csv")

print(f"Forme de df_train : {df_train.shape}")
print(f"Forme de df_test  : {df_test.shape}")
df_train.head(3)

## 3. Data Cleaning & Feature Engineering Initial
Dans cette étape, nous effectuons le nettoyage complet :
1. Conversion des dates (`trans_date_trans_time`, `dob`).
2. Conversion des montants de Dollars en **Dirhams Marocains (MAD / DH)**.
3. Calcul de l'âge du client.
4. Calcul de la **Distance Haversine (km)** entre le client et le commerçant.
5. Vérification des valeurs manquantes et des doublons.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 # Rayon moyen de la Terre en km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2.0)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def clean_and_enhance(df):
    df = df.copy()
    # Dates & Heures
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["dob"] = pd.to_datetime(df["dob"])
    df["hour"] = df["trans_date_trans_time"].dt.hour
    df["day_name"] = df["trans_date_trans_time"].dt.day_name()
    
    # Âge du client
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365
    
    # Montant en MAD
    df["amt_mad"] = df["amt"] * USD_TO_MAD
    
    # Distance Haversine (km)
    df["distance_km"] = haversine_distance(df["lat"], df["long"], df["merch_lat"], df["merch_long"])
    return df

df_train_clean = clean_and_enhance(df_train)
print("Vérification des Valeurs Manquantes :", df_train_clean.isnull().sum().sum())
print("Vérification des Doublons :", df_train_clean.duplicated(subset=["trans_num"]).sum())
df_train_clean[["trans_date_trans_time", "amt_mad", "age", "distance_km", "is_fraud"]].head()

## 4. Analyse Exploratoire & Visualisations pour le Rapport

### Figure 1 : Déséquilibre de la Variable Cible (`is_fraud`)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = df_train_clean["is_fraud"].value_counts()
pcts = df_train_clean["is_fraud"].value_counts(normalize=True) * 100

bars = ax.bar(["Légitime (0)", "Fraude (1)"], counts, color=["#2b5c8f", "#d9534f"])
ax.set_yscale("log")
ax.set_title("Répartition des Transactions (Échelle Logarithmique)", fontsize=13, fontweight="bold")
ax.set_ylabel("Nombre de Transactions (Log)")

for bar, pct, cnt in zip(bars, pcts, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.2, f"{cnt:,} ({pct:.2f}%)", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

### Figure 2 : Comparaison des Montants en Dirhams (MAD)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(np.log1p(df_train_clean[df_train_clean["is_fraud"]==0]["amt_mad"]), label="Légitime", color="#2b5c8f", fill=True, alpha=0.5)
sns.kdeplot(np.log1p(df_train_clean[df_train_clean["is_fraud"]==1]["amt_mad"]), label="Fraude", color="#d9534f", fill=True, alpha=0.6)
plt.title("Distribution des Montants (Log DH) : Transactions Légitimes vs Fraude", fontsize=13, fontweight="bold")
plt.xlabel("Log(Montant en DH + 1)")
plt.ylabel("Densité")
plt.legend()
plt.show()

stats_mad = df_train_clean.groupby("is_fraud")["amt_mad"].agg(["mean", "median", "min", "max"]).reset_index()
stats_mad["is_fraud"] = stats_mad["is_fraud"].map({0: "Légitime", 1: "Fraude"})
print("=== STATISTIQUES FINANCIÈRES EN DIRHAMS (MAD) ===")
print(stats_mad.to_string(index=False))

### Figure 3 : Taux de Fraude par Catégorie de Commerçant

In [ ]:
cat_summary = df_train_clean.groupby("category")["is_fraud"].agg(["count", "mean"]).reset_index()
cat_summary["pct_fraud"] = cat_summary["mean"] * 100
cat_summary = cat_summary.sort_values(by="pct_fraud", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x="pct_fraud", y="category", data=cat_summary, palette="Reds_r")
plt.title("Taux de Fraude par Catégorie de Commerce (%)", fontsize=13, fontweight="bold")
plt.xlabel("Pourcentage de Fraude (%)")
plt.ylabel("Catégorie")
plt.tight_layout()
plt.show()

### Figure 4 : Taux de Fraude par Heure de la Journée

In [ ]:
hourly = df_train_clean.groupby("hour")["is_fraud"].mean().reset_index()
hourly["pct_fraud"] = hourly["is_fraud"] * 100

plt.figure(figsize=(10, 4))
plt.plot(hourly["hour"], hourly["pct_fraud"], marker="o", color="#d9534f", linewidth=2.5)
plt.title("Taux de Fraude selon l'Heure de la Journée (%)", fontsize=13, fontweight="bold")
plt.xlabel("Heure (0h à 23h)")
plt.ylabel("Taux de Fraude (%)")
plt.xticks(range(0, 24))
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Synthèse & Conclusion pour le Rapport
- **Data Cleaning** : Zéro valeur manquante, zéro doublon transactionnel identifié.
- **Encodages & Transformations** : Montants convertis en Dirhams (MAD), distance géodésique Haversine ajoutée, âge calculé.
- **Visualisations** : Toutes les figures sont prêtes et sauvegardées pour votre rapport de stage dans `reports/figures/`.